# Markdown management: clearing seasonal inventory

A retailer starts with $x_1 = 160$ units of a seasonal item and has four selling months before whatever is left is cleared at a salvage price $r = 5$. Demand falls over the season (later months have lower demand curves), and marginal cost is treated as zero — the inventory is already paid for — so the goal is simply to **maximize revenue**.

We work through three versions, following Phillips, Section 12.2:

1. **A single price** held for all four months.
2. **A full markdown schedule** — a separate price each month (constrained downward in practice). A schedule beats the single price by letting the retailer charge more early and discount later as inventory and demand wind down.
3. **Adaptive re-optimization** — after month 1's sales come in different from the forecast, we re-solve for the remaining months given the updated inventory.

The single-vs-schedule gain here is about $7\%$, matching the sweater example (Example 12.1) in the text.

## Setup

Starting inventory, salvage price, and the four monthly demand curves. Each $d_t(p)$ is linear in price and floored at zero; the intercepts decline month to month, reflecting fading seasonal demand.

In [1]:
import numpy as np
from scipy.optimize import minimize_scalar, minimize


In [2]:
# Initial inventory
x_1 = 160 

# Sale price after the fourth month
r = 5 

# Demand functions
def d_1(p):
    return max(120 - 1.5 * p, 0)

def d_2(p):
    return max(90 - 1.5 * p, 0)

def d_3(p):
    return max(80 - 1.5 * p, 0)

def d_4(p):
    return max(50 - 2 * p, 0)

## Model 1: a single price for the whole season

One price $p$ is charged in all four months. Each month sells the smaller of demand and remaining inventory; the inventory rolls forward, and anything left after month 4 is salvaged at $r$. We maximize total revenue over the single price.

In [3]:
def total_revenue(p):
    # Sell min(demand, inventory) each month, rolling inventory forward
    q1 = min(d_1(p), x_1)
    x_2 = x_1 - q1
    q2 = min(d_2(p), x_2)
    x_3 = x_2 - q2
    q3 = min(d_3(p), x_3)
    x_4 = x_3 - q3
    q4 = min(d_4(p), x_4)

    q5 = x_1 - q1 - q2 - q3 - q4   # leftover, salvaged at r

    revenue = q1*p + q2*p + q3*p + q4*p + q5*r
    return -revenue   # negate: maximize via a minimizer

In [4]:
res = minimize_scalar(total_revenue, bounds=(0, 120/1.5), method='bounded')

print('The optimal price is:', np.round(res.x, 2))
print('The maximum revenue is:', round(-res.fun))


The optimal price is: 34.72
The maximum revenue is: 4775


## Model 2: a full markdown schedule

Now each month gets its own price $(p_1, p_2, p_3, p_4)$. The inventory dynamics are identical; only the pricing flexibility changes. Compare the revenue and the percent improvement over the single price.

In [5]:
def total_revenues(prices):
    p1, p2, p3, p4 = prices
    q1 = min(d_1(p1), x_1)
    x_2 = x_1 - q1
    q2 = min(d_2(p2), x_2)
    x_3 = x_2 - q2
    q3 = min(d_3(p3), x_3)
    x_4 = x_3 - q3
    q4 = min(d_4(p4), x_4)

    q5 = x_1 - q1 - q2 - q3 - q4   # leftover, salvaged at r

    revenue = q1*p1 + q2*p2 + q3*p3 + q4*p4 + q5*r
    return -revenue

p0 = [20, 20, 20, 20]
res2 = minimize(total_revenues, p0, bounds=[(0, 120/1.5)]*4)

print('The optimal prices are:', np.round(res2.x, 2))
print('The maximum revenue is:', round(-res2.fun))

The optimal prices are: [42.5  32.5  29.17 15.  ]
The maximum revenue is: 5120


In [6]:
print('Percent improvement to revenues: ', np.round((res2.fun / res.fun - 1) * 100, 2), 'percent.' )

Percent improvement to revenues:  7.21 percent.


## Model 3: re-optimizing after month 1

The schedule above is a plan, not a commitment. Suppose month 1 is priced at its planned $p_1$ but sells $70$ units instead of the forecast $\approx 56$. The retailer now has less inventory than expected, so it should re-solve for months 2–4 given the new starting stock — exactly the adaptive step Phillips describes. We lock in the realized month-1 revenue and re-optimize the rest.

In [7]:
p1_optimal = res2.x[0]   # planned month-1 price from Model 2

print('Optimal price for the first month from the previous steps: ', np.round(p1_optimal, 2))
print('At that price, you estimated to sell ', round(min(d_1(p1_optimal), x_1)), 'units.')

q1_realized = 70
print('Say you ended up selling ', q1_realized, 'units.')

Optimal price for the first month from the previous steps:  42.5
At that price, you estimated to sell  56 units.
Say you ended up selling  70 units.


In [8]:
revenue_month_1 = q1_realized * p1_optimal
x_2 = x_1 - q1_realized   # inventory carried into month 2 after realized sales

def total_revenues_updated(prices):
    p2, p3, p4 = prices
    q2 = min(d_2(p2), x_2)
    x_3 = x_2 - q2
    q3 = min(d_3(p3), x_3)
    x_4 = x_3 - q3
    q4 = min(d_4(p4), x_4)

    q5 = x_2 - q2 - q3 - q4   # leftover, salvaged at r

    revenue = q2*p2 + q3*p3 + q4*p4 + q5*r
    return -revenue

p0 = [30, 30, 10]
res = minimize(total_revenues_updated, p0, bounds=[(0, 90/1.5), (0, 80/1.5), (0, 50/2)])

total_max_revenue = revenue_month_1 - res.fun   # realized month 1 + re-optimized months 2-4

print('The optimal prices are:', np.round(res.x, 2))
print('The maximum revenue is:', round(total_max_revenue))

The optimal prices are: [34.   30.67 16.5 ]
The maximum revenue is: 5624
